# 06 The Sequence Model

Logistic regression and the MLP are **permutation invariant** over primes: shuffle the 1000 columns and retrain, and nothing changes. But the discriminative signal oscillates *smoothly* in $n$ (the PC1 weights, HLOP Fig 5), so adjacent primes carry related information that a bag-of-features model cannot pool. A 1-D CNN convolving along the prime axis is the natural model — for two distinct reasons, worth keeping separate:

1. **Weight sharing.** One kernel computes the *same* local statistic at every position; an MLP must rediscover it at all 1000 coordinates. This helps even if the primes were in arbitrary order.
2. **Locality.** Pooling neighbours raises SNR — per curve the class-mean separation ($\sim p^{0.2}$; the growth of HLOP's fitted amplitudes, Table 2) is tiny next to the spread ($\sim \sqrt{p}$, the Hasse scale) — but only if neighbouring primes actually carry related signal.

Notebook 05 left this notebook two questions: can the CNN beat the **0.987** linear baseline on real curves, and — via the pre-registered **shuffle control** — is any CNN advantage due to sequence structure (locality), or merely weight sharing? Both are answered below.

In [1]:
%load_ext autoreload
%autoreload 2
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from murmurations import ml
from murmurations import torch_models as tm
from murmurations.data import feature_matrix
from murmurations.features import moment_channels, moment_feature_matrix
from murmurations.synthetic import make_synthetic_dataset

# LogisticRegression occasionally stops just short of convergence on the
# 3000-column moment features; accuracies are stable, so silence only that.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

DATA = Path("..") / "data"
device = tm.get_device()
print("device", device)

device mps


## Input layout

`moment_channels` returns **(N, C, P)** — moments as channels, primes as the sequence axis — versus `moment_feature_matrix`'s flat (N, C·P). Given Notebook 05's verdict (the skew contributes nothing measurable on real curves), the **single raw channel** $\tilde a_p$ is the honest test; the 3-moment-channel variant is retained as an *ablation*, not a requirement.

In [2]:
df = make_synthetic_dataset(
    n_per_rank=5000, num_primes=200, mean_amp=0.0, skew_amp=2.0, seed=5
)
Xc = moment_channels(df, num_primes=200)
print("moment_channels (N, C, P):", Xc.shape)
m = tm.ApCNN(in_channels=3, n_classes=2)
print("params:", f"{sum(p.numel() for p in m.parameters()):,}")

moment_channels (N, C, P): (10000, 3, 200)
params: 52,610


## Head-to-head, identical data
Skew-only regime — the one where the MLP failed in Notebook 05.

In [3]:
y_syn = df["rank"].to_numpy()
NPR = 200


def run(X, y, model_fn, tag, epochs=60, standardize=False, reps=1):
    """
    Train model_fn(X) reps times (fresh split + init each rep); print
    val accuracy (mean +- spread when reps > 1). Split precedes scaling so
    the scaler never sees validation rows.
    """
    accs = []
    for rep in range(reps):
        Xtr, Xva, ytr, yva = train_test_split(
            X, y, test_size=0.2, random_state=rep, stratify=y
        )
        if standardize:
            scaler = StandardScaler().fit(Xtr)
            Xtr, Xva = scaler.transform(Xtr), scaler.transform(Xva)
        tl, vl = tm.make_loaders_from_arrays(
            np.ascontiguousarray(Xtr),
            ytr,
            np.ascontiguousarray(Xva),
            yva,
            batch_size=256,
            device=device,
        )
        h = tm.train(
            model_fn(X),
            tl,
            vl,
            device,
            epochs=epochs,
            lr=1e-3,
            patience=10,
            scheduler="plateau",
            verbose=False,
        )
        accs.append(max(h.val_acc))
    if reps == 1:
        print(f"  {tag:30s} {accs[0]:.3f}")
    else:
        print(
            f"  {tag:30s} {np.mean(accs):.3f} +- {np.std(accs):.3f}"
            f"  ({reps} reps)"
        )
    return accs


acc_raw = ml.repeat_logistic(df, [0, 1], seeds=range(3), n_features=NPR)[
    "acc_mean"
]
print(f"  {'logistic(raw a_p)':30s} {acc_raw:.3f}")

acc_mom = ml.repeat_logistic(
    df,
    [0, 1],
    seeds=range(3),
    n_features=NPR,
    featurizer=moment_feature_matrix,
)["acc_mean"]
print(f"  {'logistic(moment feats)':30s} {acc_mom:.3f}")

_ = run(
    feature_matrix(df, num_primes=NPR),
    y_syn,
    lambda X: tm.MLP(X.shape[1], 2, hidden=(256, 128), p_drop=0.3),
    "MLP(raw a_p)",
    standardize=True,
)
_ = run(
    moment_feature_matrix(df, num_primes=NPR),
    y_syn,
    lambda X: tm.MLP(X.shape[1], 2, hidden=(256, 128), p_drop=0.3),
    "MLP(moment feats)",
    standardize=True,
)
_ = run(
    moment_channels(df, num_primes=NPR)[:, :1, :],
    y_syn,
    lambda X: tm.ApCNN(1, 2),
    "CNN(1 chan: tilde a_p)",
)
_ = run(
    moment_channels(df, num_primes=NPR),
    y_syn,
    lambda X: tm.ApCNN(3, 2),
    "CNN(3 moment channels)",
)

  logistic(raw a_p)              0.492
  logistic(moment feats)         0.963
  MLP(raw a_p)                   0.522
  MLP(moment feats)              0.968
  CNN(1 chan: tilde a_p)         0.970
  CNN(3 moment channels)         0.969


The CNN reaches ~0.97 from a **single raw channel** — the very signal the MLP never finds at any capacity (~chance above). Weight sharing does the work: "compute a local statistic of $\tilde a_p$, pool it over primes" is one shared kernel for the CNN and 200 separate discoveries for the MLP.

Adding the moment channels buys essentially nothing (both CNN variants land ~0.97; the difference is within single-run noise). So Notebook 05's slogan *"feature engineering beats architecture"* was **MLP-specific** — the right architecture makes the hand-built features nearly redundant. Whether any of this survives on real curves, where the mean signal dominates and the linear baseline is already 0.987, is the next section.

## Real data — rank 0 vs 1, conductor [7500, 10000]

The bar from Notebook 05: logistic regression on raw $a_p$ at **0.987**. CNN accuracies below are mean $\pm$ spread over 3 repetitions (fresh split and initialization each time) — single CNN runs on this data wobble by a few hundredths, which matters for the comparisons that follow.

In [4]:
real = pd.read_parquet(DATA / "ec_7500_10000_r012_1000ap.parquet")
r01 = real[real["rank"].isin([0, 1])].copy()
y_real = r01["rank"].to_numpy()
print(r01.groupby("rank").size())

rank
0    4328
1    5194
dtype: int64


In [5]:
base = ml.repeat_logistic(r01, [0, 1], seeds=range(3), n_features=1000)
print(f"  {'logistic(raw a_p)':30s} {base['acc_mean']:.3f}")

X1 = moment_channels(r01, num_primes=1000)[:, :1, :]
X3 = moment_channels(r01, num_primes=1000)

_ = run(X1, y_real, lambda X: tm.ApCNN(1, 2), "CNN(1 chan)", epochs=40, reps=3)
_ = run(
    X3,
    y_real,
    lambda X: tm.ApCNN(3, 2),
    "CNN(3 moment channels)",
    epochs=40,
    reps=3,
)

  logistic(raw a_p)              0.987
  CNN(1 chan)                    0.915 +- 0.005  (3 reps)
  CNN(3 moment channels)         0.913 +- 0.009  (3 reps)


The CNN **loses to the linear baseline** on real curves (0.915 $\pm$ 0.005 vs 0.987) — and the moment-channel ablation confirms Notebook 05's verdict here too (0.913 $\pm$ 0.009; the extra channels change nothing). Given that verdict this is not shocking: the mean signal dominates, and logistic regression on 1000 raw coefficients already reads it near-optimally. But it does put the "sequence model" framing on notice: if the CNN cannot use the ordering to *gain* anything, is it using the ordering at all? That is exactly what the shuffle control tests.

One caveat on the comparison: the logistic baseline balances classes (0.5 base rate), while the CNN trains on the natural 4328/5194 proportions (0.545 base rate). The ~0.07 gap dwarfs that 0.045 base-rate difference, so the conclusion stands, but the comparison is not perfectly matched.

In [6]:
# Pre-registered in Notebook 05: shuffle the prime axis with a fixed
# permutation and retrain. Logistic is permutation invariant (permuting
# columns permutes coefficients; the L2 penalty is symmetric), so it is the
# negative control -- its accuracy must not move. If the CNN's performance
# is genuinely about sequence structure, shuffling must destroy it.
rng = np.random.default_rng(0)
perm = rng.permutation(1000)

shuf_logistic = ml.repeat_logistic(
    r01,
    [0, 1],
    seeds=range(3),
    n_features=1000,
    featurizer=lambda d, num_primes: feature_matrix(d, num_primes=num_primes)[
        :, perm
    ],
)["acc_mean"]
print(f"  {'logistic(raw, shuffled)':30s} {shuf_logistic:.3f}")

_ = run(
    X1[:, :, perm],
    y_real,
    lambda X: tm.ApCNN(1, 2),
    "CNN(1 chan, shuffled)",
    epochs=40,
    reps=3,
)

  logistic(raw, shuffled)        0.987
  CNN(1 chan, shuffled)          0.721 +- 0.012  (3 reps)


The negative control passes: logistic is unchanged at 0.987, confirming the shuffle touches only the ordering. The CNN **collapses from 0.915 $\pm$ 0.005 to 0.721 $\pm$ 0.012** — a drop of ~0.19 against a spread of ~0.01.

The CNN *is* using the sequence structure — locality is real on real curves, exactly as the smooth PC1 oscillation predicts. Its problem is not the inductive bias but redundancy: the ordering information the CNN exploits is already contained in what the linear model extracts from the unordered coefficients. Knowing the ordering doesn't add accuracy over the linear baseline; destroying it subtracts a fifth of the CNN's.

One detail worth noting: even shuffled, the CNN reaches 0.72 — Far above the 0.55 majority-class floor (4328/5194 class split). That residue is weight sharing operating on an unordered bag (plus whatever per-prime marginal signal survives pooling), which is precisely the decomposition the synthetic experiments predicted: weight sharing gives the CNN a floor; locality, when the signal oscillates across the input, provides the rest.

## Knobs worth turning
- **Receptive field.** Three `MaxPool1d(4)` layers downsample 64x, so 1000 primes -> ~15. Fig 5's oscillation has a period of *hundreds* of primes, so the field must be wide. Try `pool=2` (less aggressive) or dilated convs.
- **Kernel size** `kernel_size=7` or 9 for a wider local window.
- **Prime axis vs prime index.** We convolve over $n$, but the Prime Number Theorem asserts $p_n \sim n\log(n)$. Resampling onto a uniform $p$ or $\sqrt p$ grid is a genuinely different model.